In [1]:
import pymysql

#create a connection
conn = pymysql.connect(
    host = 'localhost',
    user = 'root',
    password = "sabi3601",
    database = 'nhl_analytics'
)

#create a cursor
cursor = conn.cursor()

In [2]:
cursor.execute("SELECT team_id, team_abbrev FROM teams")

team_abbrev_to_id = {abbrev: team_id for team_id, abbrev in cursor.fetchall()}

team_abbrevs = list(team_abbrev_to_id.keys())

len(team_abbrevs)

32

In [3]:
import requests

url = f"https://api-web.nhle.com/v1/club-schedule-season/{team_abbrevs[0]}/20252026"

response = requests.get(url)

response.status_code # 200 indicates success

200

In [4]:
data = response.json()
data.keys()

dict_keys(['previousSeason', 'currentSeason', 'nextSeason', 'clubTimezone', 'clubUTCOffset', 'games'])

In [5]:
len(data['games'])

101

In [7]:
data['games'][0]

{'id': 2025010001,
 'season': 20252026,
 'gameType': 1,
 'gameDate': '2025-09-21',
 'venue': {'default': 'Toyota Arena'},
 'neutralSite': True,
 'startTimeUTC': '2025-09-21T22:00:00Z',
 'easternUTCOffset': '-04:00',
 'venueUTCOffset': '-07:00',
 'venueTimezone': 'America/Los_Angeles',
 'gameState': 'FINAL',
 'gameScheduleState': 'OK',
 'tvBroadcasts': [],
 'awayTeam': {'id': 24,
  'commonName': {'default': 'Ducks'},
  'placeName': {'default': 'Anaheim'},
  'placeNameWithPreposition': {'default': 'Anaheim', 'fr': "d'Anaheim"},
  'abbrev': 'ANA',
  'logo': 'https://assets.nhle.com/logos/nhl/svg/ANA_light.svg',
  'darkLogo': 'https://assets.nhle.com/logos/nhl/svg/ANA_dark.svg',
  'awaySplitSquad': False,
  'score': 1},
 'homeTeam': {'id': 26,
  'commonName': {'default': 'Kings'},
  'placeName': {'default': 'Los Angeles'},
  'placeNameWithPreposition': {'default': 'Los Angeles',
   'fr': 'de Los Angeles'},
  'abbrev': 'LAK',
  'logo': 'https://assets.nhle.com/logos/nhl/svg/LAK_light.svg',


In [8]:
import time

games_by_id = {}

for abbrev in team_abbrevs:

    url = f"https://api-web.nhle.com/v1/club-schedule-season/{abbrev}/20252026"
    response = requests.get(url)
    schedule = response.json()

    for g in schedule['games']:
        gid = g['id']
        if gid in games_by_id:
            continue

        games_by_id[gid] = {
            'game_id': gid,
            'season': g['season'],
            'game_type': g['gameType'],
            'game_date': g['gameDate'],
            'home_team_abbrev': g['homeTeam']['abbrev'],
            'away_team_abbrev': g['awayTeam']['abbrev'],
            'home_score': g['homeTeam'].get('score'),
            'away_score': g['awayTeam'].get('score'),
            'game_state': g['gameState'],
            'venue_name': g['venue']['default']
        }

    time.sleep(0.5)

games = list(games_by_id.values())

print(f"{len(games)} unique games collected")

1498 unique games collected


In [9]:
create_games_table = """
CREATE TABLE IF NOT EXISTS games (
    game_id BIGINT PRIMARY KEY,
    season VARCHAR(20),
    game_type INT,
    game_date DATE,
    home_team_id INT,
    away_team_id INT,
    home_score INT,
    away_score INT,
    game_state VARCHAR(20),
    venue_name VARCHAR(150),
    FOREIGN KEY (home_team_id) REFERENCES teams(team_id),
    FOREIGN KEY (away_team_id) REFERENCES teams(team_id)
)
"""

cursor.execute(create_games_table)

conn.commit()

print("Games table created successfully")

Games table created successfully


In [10]:
for row in games:
    row['home_team_id'] = team_abbrev_to_id[row['home_team_abbrev']]
    row['away_team_id'] = team_abbrev_to_id[row['away_team_abbrev']]

games[0]

{'game_id': 2025010001,
 'season': 20252026,
 'game_type': 1,
 'game_date': '2025-09-21',
 'home_team_abbrev': 'LAK',
 'away_team_abbrev': 'ANA',
 'home_score': 3,
 'away_score': 1,
 'game_state': 'FINAL',
 'venue_name': 'Toyota Arena',
 'home_team_id': 20,
 'away_team_id': 18}

In [11]:
insert_query = """INSERT IGNORE INTO games
                    (game_id,season,game_type,game_date,home_team_id,away_team_id,home_score,away_score,game_state,venue_name)
                    VALUES(%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)"""

for i in games:
   values = (i['game_id'],
         i['season'],
         i['game_type'],
         i['game_date'],
         i['home_team_id'],
         i['away_team_id'],
         i['home_score'],
         i['away_score'],
         i['game_state'],
         i['venue_name'])
   cursor.execute(insert_query,values)

conn.commit()

print(f"{len(games)} games inserted successfully")

1498 games inserted successfully


In [12]:
cursor.execute("SELECT COUNT(*) FROM games")
cursor.fetchone()

(1498,)